In [2]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/"

# import os
# os.chdir(r"C:\Z")  # Cambia el directorio de trabajo
# print(os.getcwd())  # Verifica que cambió correctamente

import os
import sys
import random
from astropy.io import fits
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [3]:
# Eliminador

pickle_folder = 'extra'
spectrums_folder = r'spectrums training 100k'

# Obtener la lista de archivos pickle que comienzan con "batch_files_list" y terminan con ".pkl"
pickle_files = [f for f in os.listdir(pickle_folder) if f.startswith('batch_files_list') and f.endswith('.pkl')]

# Recorrer cada archivo pickle
for pickle_file in pickle_files:
    pickle_path = os.path.join(pickle_folder, pickle_file)
    print(f"Procesando {pickle_path}...")
    try:
        with open(pickle_path, 'rb') as f:
            file_list = pickle.load(f)
    except Exception as e:
        print(f"Error abriendo {pickle_path}: {e}")
        continue
    
    # Recorrer la lista de nombres y eliminar los archivos correspondientes en la carpeta spectrums
    for filename in file_list:
        file_path = os.path.join(spectrums_folder, filename)
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
                print(f"Eliminado: {file_path}")
            except Exception as e:
                print(f"Error eliminando {file_path}: {e}")
        else:
            print(f"Archivo no encontrado: {file_path}")

print("Proceso de eliminación completado.")

Proceso de eliminación completado.


In [ ]:
# Procesamiento de archivos completo

# APLICACIÓN A LOS ARCHIVOS DE LA CARPETA SPECTRUMS
folder_path = r'spectrums training 100k'
batch_size = 100000  # Número de archivos a procesar por lote
index_file = "extra/batch_index.txt"

# Verificar que el archivo batch_index.txt exista
if not os.path.exists(index_file):
    print(f"Error: El archivo {index_file} no existe. Deteniendo la ejecución.")
    sys.exit(1)

# Intentar leer y convertir el contenido a un número entero
with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: El archivo {index_file} no contiene un número entero válido. Deteniendo la ejecución.")
        sys.exit(1)

# Función para expandir puntos (asegúrate de tener definida esta función)
def expand_points(wavelength, flux, target_count=5000):
    # Ejemplo de interpolación lineal
    new_wavelength = np.linspace(wavelength.min(), wavelength.max(), target_count)
    new_flux = np.interp(new_wavelength, wavelength, flux)
    return new_wavelength, new_flux

# Obtener la lista inicial de archivos FITS
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]

while files:
    # Seleccionar aleatoriamente el lote actual (si quedan menos de batch_size, se toman todos)
    if len(files) < batch_size:
        current_batch = files.copy()
    else:
        current_batch = random.sample(files, batch_size)
    
    # Diccionario de almacenamiento para el lote actual
    spectra_data = {}
    i = 0

    for filename in current_batch:
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                # Verifica que el archivo tenga las extensiones esperadas
                if len(hdul) > 2:
                    flux_data = hdul[1].data["flux"]         # Datos de flujo
                    loglam_data = hdul[1].data["loglam"]        # Datos de log(lambda)
                    wavelength_data = 10 ** loglam_data         # Convertir log(lambda) a longitud de onda
                    redshift = hdul[2].data["Z"][0]             # Extraer redshift
                    
                    # Aplicar la interpolación para obtener 5000 puntos
                    interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                    
                    # Guardar los datos
                    spectra_data[filename] = {
                        "wavelength": interp_wavelength,
                        "flux": interp_flux,
                        "redshift": redshift
                    }
                    
                    i += 1
                    if i % 1000 == 0:
                        print(f"Procesado {filename} ({i})")
        except Exception as e:
            print(f"Error procesando {filename}: {e}")
    
    print(f"Se procesaron {len(spectra_data)} archivos FITS en el lote {batch_index}.")
    
    # Guardar el diccionario en un archivo pickle llamado spectra_data_complete{batch_index}.pkl
    output_file = f'data/spectra_data_complete{batch_index}.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Datos guardados en {output_file}")
    
    # Guardar la lista de archivos procesados en este lote
    batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
    with open(batch_list_file, 'wb') as f:
        pickle.dump(current_batch, f)
    print(f"Lista de archivos del lote {batch_index} guardada en {batch_list_file}")
    
    # Actualizar la lista de archivos: eliminar los procesados
    files = [f for f in files if f not in current_batch]
    
    # Actualizar el archivo batch_index.txt para la siguiente iteración
    batch_index += 1
    with open(index_file, "w") as f:
        f.write(str(batch_index))
    
print("Se han procesado y eliminado todos los archivos de la carpeta.")

Procesado spec-4635-55615-0354.fits (1000)
Procesado spec-10267-58223-0029.fits (2000)
Procesado spec-6145-56266-0076.fits (3000)
Procesado spec-8528-57896-0308.fits (4000)
Procesado spec-11654-58543-0536.fits (5000)
Procesado spec-4494-55569-0728.fits (6000)
Procesado spec-6125-56273-0962.fits (7000)
Procesado spec-5414-56014-0837.fits (8000)
Procesado spec-6137-56270-0777.fits (9000)
Procesado spec-1250-52930-0304.fits (10000)
Procesado spec-2435-53828-0548.fits (11000)
Procesado spec-1814-54555-0346.fits (12000)
Procesado spec-4758-55682-0644.fits (13000)
Procesado spec-3822-55544-0902.fits (14000)
Procesado spec-6718-56398-0132.fits (15000)
Procesado spec-5487-55982-0596.fits (16000)
Procesado spec-2035-53436-0509.fits (17000)
Procesado spec-10263-58464-0970.fits (18000)
Procesado spec-0855-52375-0430.fits (19000)
Procesado spec-4743-55645-0721.fits (20000)
Procesado spec-8381-57512-0582.fits (21000)
Procesado spec-7904-57358-0291.fits (22000)
Procesado spec-6302-56565-0079.fits (2

In [ ]:
# TEST EXTRACCIÓN FITS

# Ruta al archivo FITS
file_path = r'spectrums/spec-0266-51630-0003.fits'

# Leer el archivo FITS
with fits.open(file_path) as hdul:
    hdul.info()  # Información general del archivo

    # print(hdul[1].columns)  # Ver columnas del HDU 1
    test_flux = hdul[1].data["flux"]
    test_loglam = hdul[1].data["loglam"]

    # print(hdul[2].columns)  # Ver columnas del HDU 2
    test_redshift = hdul[2].data["Z"]
    print("Redshift:", test_redshift)
    test_class = hdul[2].data["CLASS"]
    print("Redshift:", test_class)

# Convertir loglam a longitud de onda (Ångstroms)
test_wavelength = 10 ** test_loglam

# Graficar el espectro
plt.figure(figsize=(12, 6))
plt.plot(test_wavelength, test_flux, label="Test Espectro")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Test)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Función para interpolación adaptativa (15000 FITS = 5hrs POCO VIABLE EN DATASETS GRANDES PREPARAR LOS DATOS)

def expand_points(wavelength, flux, target_count=5000):
    # Convertimos a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Continuamos insertando hasta alcanzar el número deseado de puntos
    while len(wl) < target_count:
        # Calcular las diferencias absolutas en flux entre puntos consecutivos
        diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl) - 1)]
        # Encontrar el índice donde la diferencia es máxima
        max_idx = np.argmax(diffs)
        # Interpolar linealmente para obtener un nuevo punto
        new_wl = (wl[max_idx] + wl[max_idx+1]) / 2
        new_fl = (fl[max_idx] + fl[max_idx+1]) / 2
        # Insertar el nuevo punto en la posición correspondiente
        wl.insert(max_idx+1, new_wl)
        fl.insert(max_idx+1, new_fl)
        
    return np.array(wl), np.array(fl)

# TEST CON EL ANTERIOR FITS
expanded_wavelength, expanded_flux = expand_points(test_wavelength, test_flux, target_count=5000)

# Graficar el espectro ampliado
plt.figure(figsize=(12, 6))
plt.plot(expanded_wavelength, expanded_flux, label="Espectro Expandido")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Expandido a 5000 puntos)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()